In [5]:
import sys
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir      = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config
import triage_filter

cfg = load_config()

# Resolve triage path (new key — created by run_triage if absent)
triage_path = Path(cfg["paths"]["triage"])

print("Config loaded.")
print("  raw_images :", cfg["paths"]["raw_images"])
print("  triage     :", triage_path)


Config loaded.
  raw_images : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw
  triage     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/triage


In [6]:
# Loads SigLIP ViT-SO400M-14-SigLIP-384 via open_clip.
# Downloads ~3 GB of weights on first run (cached by HuggingFace Hub).
model, processor, device = triage_filter.load_siglip_model(cfg)
print(f"SigLIP ready on: {device}")


open_clip_config.json:   0%|          | 0.00/921 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[SigLIP] Loaded ViT-SO400M-14-SigLIP-384 on cuda
SigLIP ready on: cuda


In [7]:
# Test mode: process only the first 10 patent folders.
# Remove limit=10 (or set limit=None) to run on the full dataset.
triage_filter.run_triage(cfg, threshold=0.60, limit=10)


[SigLIP] Loaded ViT-SO400M-14-SigLIP-384 on cuda
  ✓ AU2021232803A1  total=10  flagged=10
  ✓ BR112023017877B1PAFP  total=7  flagged=7
  ✓ CA2963502A1  total=2  flagged=2
  ✓ CN103786881APAFP  total=12  flagged=12
  ✓ CN104229137APAFP  total=9  flagged=9
  ✓ CN105799934APAFP  total=9  flagged=9
  ✓ CN105923154APAFP  total=2  flagged=2
  ✓ CN106005373APAFP  total=12  flagged=12
  ✓ CN106043685APAFP  total=3  flagged=3
  ✓ CN106314794APAFP  total=7  flagged=7

Triage complete: 10 patents | 73 images | 73 flagged (100.0%)
Summary written to: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/triage/triage_summary.csv


In [8]:
import json
import pandas as pd

triage_path = Path(cfg["paths"]["triage"])
summary_csv = triage_path / "triage_summary.csv"

if not summary_csv.exists():
    print("No triage_summary.csv found — run Cell 3 first.")
else:
    summary_df = pd.read_csv(summary_csv)

    total_images  = summary_df["total_images"].sum()
    total_flagged = summary_df["flagged_count"].sum()
    overall_ratio = total_flagged / max(1, total_images)

    print(f"Total images scored : {total_images}")
    print(f"Total flagged       : {total_flagged}")
    print(f"Overall flagged ratio: {overall_ratio:.1%}")
    print()

    # Show per-figure results for the 3 patents with the most flagged images
    top3 = summary_df.nlargest(3, "flagged_count")[["patent_id", "total_images", "flagged_count", "flagged_ratio"]]
    print("Top 3 patents by flagged count:")
    display(top3.reset_index(drop=True))
    print()

    for patent_id in top3["patent_id"]:
        json_path = triage_path / f"{patent_id}.json"
        if not json_path.exists():
            print(f"  {patent_id}: JSON not found")
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        figures_df = pd.DataFrame(data["figures"])
        print(f"\n--- {patent_id}  (total={data['total']}  flagged={data['flagged']}) ---")
        display(figures_df[["file", "pred", "score_drawing", "score_table", "keep"]])


Total images scored : 73
Total flagged       : 73
Overall flagged ratio: 100.0%

Top 3 patents by flagged count:


,patent_id,total_images,flagged_count,flagged_ratio
0,CN103786881APAFP,12,12,1.0
1,CN106005373APAFP,12,12,1.0
2,AU2021232803A1,10,10,1.0




--- CN103786881APAFP  (total=12  flagged=12) ---


,file,pred,score_drawing,score_table,keep
0,140228190200.png,drawing,0.5190,0.4810,False
1,140228190204.png,drawing,0.5177,0.4823,False
2,140228190207.png,drawing,0.5147,0.4853,False
3,140228190301.png,drawing,0.5237,0.4763,False
4,140228190305.png,drawing,0.5235,0.4765,False
5,140228190308.png,drawing,0.5229,0.4771,False
6,140228192311.png,drawing,0.5100,0.4900,False
7,140228192314.png,drawing,0.5207,0.4793,False
8,140228192318.png,drawing,0.5098,0.4902,False
9,140228195546.png,drawing,0.5122,0.4878,False



--- CN106005373APAFP  (total=12  flagged=12) ---


,file,pred,score_drawing,score_table,keep
0,160522144736.png,drawing,0.5079,0.4921,False
1,160522144745.png,drawing,0.5113,0.4887,False
2,160522144754.png,drawing,0.5098,0.4902,False
3,160522144802.png,drawing,0.5102,0.4898,False
4,160522144810.png,drawing,0.5111,0.4889,False
5,160522144822.png,drawing,0.5111,0.4889,False
6,160522144831.png,drawing,0.5102,0.4898,False
7,160522144839.png,drawing,0.5084,0.4916,False
8,160522144851.png,drawing,0.5072,0.4928,False
9,160522144900.png,drawing,0.5101,0.4899,False



--- AU2021232803A1  (total=10  flagged=10) ---


,file,pred,score_drawing,score_table,keep
0,AU2021232803A1_img1.png,drawing,0.5281,0.4719,False
1,AU2021232803A1_img10.png,drawing,0.5266,0.4734,False
2,AU2021232803A1_img2.png,drawing,0.5237,0.4763,False
3,AU2021232803A1_img3.png,drawing,0.5178,0.4822,False
4,AU2021232803A1_img4.png,drawing,0.5300,0.4700,False
5,AU2021232803A1_img5.png,drawing,0.5204,0.4796,False
6,AU2021232803A1_img6.png,drawing,0.5307,0.4693,False
7,AU2021232803A1_img7.png,drawing,0.5311,0.4689,False
8,AU2021232803A1_img8.png,drawing,0.5182,0.4818,False
9,AU2021232803A1_img9.png,drawing,0.5180,0.4820,False
